# Pull team-stint splits for traded players

nba_api's season-total stats mislabel a traded player's whole season under only their last team, so this uses the per-game `PlayerGameLogs` endpoint to rebuild corrected per-team-stint totals for anyone who played for 2+ teams in a season. Outputs `traded_player_stints.csv`.

In [ ]:
import pandas as pd
import time
import os
from nba_api.stats.endpoints import playergamelogs

REQUEST_TIMEOUT = 60
MAX_RETRIES = 4
RETRY_BACKOFF_BASE = 2.0

SEASONS = ["2017-18","2018-19","2019-20","2020-21","2021-22","2022-23","2023-24","2024-25","2025-26"]

def season_label_year(season):
    return int(season.split("-")[0])

def pull_game_logs(season, season_type="Regular Season"):
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = playergamelogs.PlayerGameLogs(
                season_nullable=season,
                season_type_nullable=season_type,
                league_id_nullable="00",
                timeout=REQUEST_TIMEOUT,
            )
            df = resp.get_data_frames()[0]
            if df.empty:
                raise ValueError(f"Empty response for season={season!r}")
            return df
        except Exception as e:
            last_exc = e
            wait = RETRY_BACKOFF_BASE * (2 ** attempt)
            print(f"  attempt {attempt+1} failed: {e!r} -- retrying in {wait:.1f}s")
            time.sleep(wait)
    raise last_exc

print("This will make one bulk API call per season (9 total) -- may take a few minutes.")

In [ ]:
box_sum_cols = ["FGM","FGA","FG3M","FG3A","FTM","FTA","OREB","DREB","AST","STL","BLK","TOV","PF","PTS","MIN"]

all_stints = []
for season in SEASONS:
    label = season_label_year(season)
    print(f"Pulling {season} game logs...")
    logs = pull_game_logs(season)
    print(f"  {len(logs)} game rows, {logs['PLAYER_ID'].nunique()} players")

    # how many distinct teams did each player suit up for this season?
    teams_per_player = logs.groupby("PLAYER_ID")["TEAM_ABBREVIATION"].nunique()
    traded_ids = teams_per_player[teams_per_player > 1].index
    print(f"  {len(traded_ids)} players played for 2+ teams")

    traded_logs = logs[logs["PLAYER_ID"].isin(traded_ids)]
    stints = traded_logs.groupby(["PLAYER_ID","PLAYER_NAME","TEAM_ABBREVIATION"])[box_sum_cols].sum().reset_index()
    stints["GP"] = traded_logs.groupby(["PLAYER_ID","PLAYER_NAME","TEAM_ABBREVIATION"]).size().values
    stints["Season"] = label
    all_stints.append(stints)

traded_all = pd.concat(all_stints, ignore_index=True)
print(f"\nTotal stint-rows across all seasons: {len(traded_all)}")
traded_all.head(10)

In [ ]:
out_path = os.path.join(os.getcwd(), "traded_player_stints.csv")
traded_all.to_csv(out_path, index=False)
print(f"Wrote {out_path}")

# sanity check: Luka 2024-25 should show two rows, DAL and LAL, GP summing close to 47
check = traded_all[(traded_all["PLAYER_NAME"].str.contains("Luka", case=False, na=False)) & (traded_all["Season"]==2024)]
check